# Volleyball

In [1]:
%pylab

Using matplotlib backend: Qt5Agg
Populating the interactive namespace from numpy and matplotlib


In [2]:
import copy

def linspc(a,b,n):
    x0=(a+b)/2
    x1=(b-a)/n*(arange(n)-(n-1)/2)
    return x0+x1

#### Class enabling the generation of patch locations

In [13]:
class point_on_sphere():
    def __init__(self,dtheta,r=1.0,verbose=0):
        self.verbose=0
        self.dtheta=dtheta
        self.set_spherical(0.0,0.0,r)
        self.to_cartesian()
    def cartesian(self,theta,phi,r):
        x=r*cos(theta)*cos(phi)
        y=r*cos(theta)*sin(phi)
        z=r*sin(theta)
        return x,y,z
    def to_cartesian(self):
        self.x,self.y,self.z=self.cartesian(self.theta,self.phi,self.r)
    def spherical(self,x,y,z):
        r=sqrt(x**2+y**2+z**2)
        theta=arcsin(z/r)
        phi=arctan2(y,x)
        return theta,phi,r
    def to_spherical(self):
        self.theta,self.phi,self.r=self.spherical(self.x,self.y,self.z)
    def set_cartesian(self,x,y,z):
        self.x=x
        self.y=y
        self.z=z
        self.to_spherical()
    def set_spherical(self,theta,phi,r):
        self.theta=theta
        self.phi=phi
        self.r=r
        self.to_cartesian()
    def go_east(self,dtheta):
        self.size=self.r*dtheta
        self.theta-=dtheta/2
        self.phi+=dtheta/cos(self.theta)
        self.theta+=dtheta/2
        self.to_cartesian()
    def go_north(self,dtheta):
        self.theta+=dtheta
        self.to_cartesian()
    def permute(self):
        new=copy.copy(self)
        new.x,new.y,new.z=self.y,self.z,self.x
        new.to_spherical()
        return new
    def info(self,label,verbose=0):
        if self.verbose>0 or verbose>0:
            print(label)
            print(' x:{:5.3f}  theta:{:5.3f}'.format(self.x,self.theta))
            print(' y:{:5.3f}    phi:{:5.3f}'.format(self.y,self.phi))
            print(' z:{:5.3f}      r:{:5.3f}'.format(self.z,self.r))
    def is_inside(self,limit):
        """ For a patch to be inside a volleyball face its central point 
            must have a latitude less that the limit, and its latitude
            with respect to the east face must be larger than the limit.
            A more careful constraint is that it is outside if *all* of
            its corners are outside the limit. The last corner to leave
            is has a position displaced by size/2 to the west, nord, and
            down.
        """
        inside=self.theta<limit             #
        corner=copy.copy(self)
        corner.go_east(-self.dtheta/2)
        corner.go_north(-self.dtheta/2)
        corner.r=corner.r-corner.size/2
        east=corner.permute()               # corner point, in east coords
        inside=inside and east.theta>limit  # mus be outside east VB face
        if self.verbose>0:
            print('inside:',inside)
        return inside
    def xyz(self):
        return self.x,self.y,self.z

#### Generate patches

In [14]:
theta0=arctan(sqrt(0.5))
dtheta=theta0/10
point=point_on_sphere(dtheta)
save=copy.copy(point)
points=[]
while(point.is_inside(theta0)):
    point.go_north(dtheta/2)
    point.go_east(dtheta/2)
    while (point.is_inside(theta0)):
        points.append(point.xyz())
        point.go_east(dtheta)
    point=save
    point.go_north(dtheta)
    save=copy.copy(point)

#### Plot patch tops and face boundaries:

In [15]:
def norm(r):
    return sqrt(dot(r,r))
def plot_top(p,size):
    rhat=p/norm(p)
    north=array([0,0,1])
    east=cross(north,rhat)
    east=east/norm(east)
    north=cross(rhat,east)
    north=north*size
    east=east*size
    face=[p+(north+east)/2,p+(north-east)/2,
          p-(north+east)/2,p-(north-east)/2,p+(north+east)/2]
    face=transpose(face)
    plot(face[1],face[2])
# Plot patch tops
clf()
face=[]
for p in points:
    face.append([+p[0],+p[1],+p[2]])   
    face.append([+p[0],-p[1],+p[2]])
    face.append([+p[0],+p[1],-p[2]])   
    face.append([+p[0],-p[1],-p[2]])
for p in face:
    plot(p[1],p[2],'+')
    plot_top(p,dtheta)
# Ploit VB face boundaries
theta=theta0
phi=linspace(-pi/4,pi/4)
plot(cos(theta)*cos(phi),cos(theta)*sin(phi))
plot(cos(phi),sin(phi))
theta=linspace(0,theta0)
phi=pi/4
plot(cos(theta)*cos(phi),cos(theta)*sin(phi));
phi=-pi/4
plot(cos(theta)*cos(phi),cos(theta)*sin(phi));

##### Complete sphere

In [17]:
figure(figsize=(8,9))
clf()
faces=[]
for p in face:
    faces.append([+p[0],+p[1],+p[2]])
    faces.append([-p[0],-p[1],-p[2]])
    p=[p[1],p[2],p[0]]
    faces.append([+p[0],+p[1],+p[2]])
    faces.append([-p[0],-p[1],-p[2]])
    p=[p[1],p[2],p[0]]
    faces.append([+p[0],+p[1],+p[2]])
    faces.append([-p[0],-p[1],-p[2]])
for p in faces:
    if p[0]>0:
        plot(p[1],p[2],'+')
        plot_top(p,dtheta)